# ЛР2. DCGAN на Fashion-MNIST: математика, интерпретация, устойчивость

Сквозной проход по работе через вызовы пакета `gan_robustness`. Тяжёлые вычисления (обучение классификатора и четырёх DCGAN, метрики, интерпретация) выполняются командой `make all`; здесь — лёгкие живые вызовы и готовые таблицы/рисунки из `reports/`.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch
from IPython.display import Image, display

from gan_robustness.config import load_config

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FIG = ROOT / 'reports' / 'figures'
TAB = ROOT / 'reports' / 'tables'
config = load_config(ROOT / 'configs' / 'base.yaml')
print('latent', config.model.latent_dim, '| g_base', config.model.g_base,
      '| subsample', config.data.subsample_size)

## Часть 1. Задача и данные

Fashion-MNIST: 60k train (10 сбалансированных классов) + 10k test, 1 канал 28×28, вход в [-1, 1]. Проверка качества чистая.

In [ ]:
display(pd.read_csv(TAB / 'part1_quality.csv'))
for name in ['part1_examples', 'part1_class_histogram', 'part1_umap_pixels']:
    display(Image(str(FIG / f'{name}.png')))

## Часть 2. Математика и архитектура DCGAN

Минимаксная игра, оптимальный дискриминатор $D^*(x)=p_{data}/(p_{data}+p_g)$, несатурирующая потеря генератора. Число параметров — живой вызов `summarize_model`.

In [ ]:
from gan_robustness.models.generator import Generator
from gan_robustness.models.discriminator import Discriminator
from gan_robustness.models.summary import summarize_model

g = Generator(config.model.latent_dim, config.model.g_base)
d = Discriminator(config.model.d_base)
display(summarize_model(g, 'Генератор'))
display(summarize_model(d, 'Дискриминатор'))

## Часть 3. Базовое обучение и оценка

Живая генерация из обученного генератора, затем сводные метрики и рисунки.

In [ ]:
from gan_robustness.training.checkpoint import load_gan
from gan_robustness.evaluation.generate import generate_float, to_uint8
from gan_robustness.visualization.gan_viz import tile_grid
import matplotlib.pyplot as plt

device = torch.device('cpu')
g_base, d_base = load_gan(ROOT / 'artifacts' / 'models' / 'baseline.pt', device)
sample = to_uint8(generate_float(g_base, 36, config.model.latent_dim, device, 0))
plt.figure(figsize=(5, 5)); plt.imshow(tile_grid(sample), cmap='gray'); plt.axis('off')
plt.title('Живая генерация (базовая модель)'); plt.show()

In [ ]:
display(pd.read_csv(TAB / 'metrics_summary.csv'))
for name in ['part3_samples_baseline', 'part3_loss_baseline',
             'part3_storyboard_baseline', 'part3_classdist_baseline',
             'part3_memorization_baseline']:
    p = FIG / f'{name}.png'
    if p.exists():
        display(Image(str(p)))

Базовая модель даёт узнаваемую одежду; D(x) и D(G(z)) держатся около устойчивого равновесия. Проверка запоминания: генерации не ближе к train, чем реальные тестовые объекты.

## Часть 4. Нарушение данных: дисбаланс и отравление

Вариант A — удаление 95% трёх классов (выпадение мод); Вариант B — триггер 4×4.

In [ ]:
for name in ['part3_classdist_imbalance', 'part3_samples_imbalance',
             'part3_classdist_poison_eps0.2', 'part3_samples_poison_eps0.2']:
    p = FIG / f'{name}.png'
    if p.exists():
        display(Image(str(p)))

При дисбалансе падает recall и доля редких классов в генерациях при сохранении precision (реалистичность). При отравлении часть генераций воспроизводит триггер (см. столбец trigger_rate в сводной таблице).

## Часть 5. Интерпретация

In [ ]:
for name in ['part5_interp_baseline', 'part5_latent_sensitivity',
             'part5_dfeatures_umap', 'part5_dfeatures_poison_umap',
             'part5_generator_activations', 'part5_discriminator_filters',
             'part5_sensitivity_trigger', 'part5_weights_discriminator',
             'part5_key_objects']:
    p = FIG / f'{name}.png'
    if p.exists():
        display(Image(str(p)))

Латентное пространство генератора гладко интерполирует между объектами; признаки дискриминатора разделяют реальные и сгенерированные объекты, а триггерные изображения образуют отдельный кластер. Grad-CAM отравленного дискриминатора фокусируется на угловом триггере.

## Выводы

Полные выводы и паспорт модели — в `reports/report.md` и `reports/model_passport.md`. DCGAN моделирует распределение данных через состязательную игру и свёрточную индуктивную гипотезу; он уязвим к дисбалансу (выпадение мод) и отравлению (воспроизведение артефакта), что видно и в метриках, и во внутренних представлениях.